# 음성 AI — STT · TTS · Realtime API

**오디오도 토큰이다.** 음성 파형을 오디오 토큰으로 변환하면 Transformer가 텍스트와 동일한 원리로 처리할 수 있습니다.
이 노트북에서는 OpenAI Audio API의 세 가지 핵심 기능을 실습합니다.

| 기능 | 엔드포인트 | 설명 |
|:---|:---|:---|
| **TTS** (음성 합성) | `audio.speech` | 텍스트 → 음성. '말하는 방식'(어조·감정)까지 프롬프트로 지시 |
| **STT** (음성 인식) | `audio.transcriptions` | 음성 → 텍스트 (받아쓰기·자막·회의록) |
| **Realtime** (실시간) | `Realtime` (WebSocket/WebRTC) | 음성을 직접 이해·생성. 낮은 지연, 끼어들기 처리 |

### 기존 파이프라인 vs 단일 모델

- **기존**: `STT -> LLM -> TTS` — 단계마다 지연이 누적되고, 어조·감정 정보가 소실됨
- **단일 모델 (Realtime)**: 음성을 직접 이해·생성 — 낮은 지연, 끼어들기(barge-in) 처리

### 모델 발전

`2022 Whisper` -> `2025 gpt-4o-transcribe / gpt-4o-mini-tts` -> `2026 GPT-Realtime-2`

> 활용처: 음성 비서, 콜센터 상담, 실시간 통역(70+ 언어)

## 1. TTS — 텍스트를 음성으로 (`audio.speech`)

텍스트를 자연스러운 음성으로 변환합니다. `voice` 로 목소리를 고르고,
`gpt-4o-mini-tts` 모델에서는 `instructions` 로 **말하는 방식(어조·감정)** 까지 지시할 수 있습니다.

- `voice`: alloy, echo, fable, onyx, nova, shimmer, coral 등
- `response_format`: mp3, wav, opus, pcm 등

### 어조·감정 지시 — `instructions`

같은 문장이라도 `instructions` 로 '어떻게 말할지'를 바꿀 수 있습니다.

## 2. STT — 음성을 텍스트로 (`audio.transcriptions`)

방금 TTS로 만든 음성 파일을 다시 텍스트로 받아쓰기 해봅니다.
받아쓰기·자막·회의록 작성에 활용됩니다.

- `gpt-4o-transcribe` / `gpt-4o-mini-transcribe`: 고품질·저지연
- `whisper-1`: 다국어 범용 모델

### 자막용 타임스탬프 (whisper-1)

`whisper-1` 모델은 `verbose_json` 포맷으로 구간별 타임스탬프를 제공해 자막(SRT) 제작에 유용합니다.

### 음성 번역 (실시간 통역의 기초)

`translations` 엔드포인트(whisper-1)는 어떤 언어의 음성이든 **영어 텍스트**로 옮깁니다.
다국어 통역 파이프라인의 출발점입니다.

In [ ]:
# 한국어 음성 -> 영어 텍스트

## 3. Realtime API — 단일 모델 음성 대화 (WebSocket)

`STT -> LLM -> TTS` 3단계 파이프라인 대신, **하나의 모델이 음성을 직접 이해하고 음성을 생성**합니다.
지연이 낮고, 사용자가 도중에 끼어들면(barge-in) 즉시 멈추고 응답할 수 있습니다.

> Realtime은 WebSocket(서버) 또는 WebRTC(브라우저)로 연결합니다.
> 아래는 텍스트 메시지를 보내 **음성 답변을 받아 파일로 저장**하는 최소 예제입니다.
> (마이크 실시간 스트리밍은 `sounddevice` 등이 추가로 필요합니다.)

In [ ]:
    # 세션 설정: 음성으로 답하도록 지정
    # 사용자 메시지(텍스트) 추가 후 응답 요청
    # 스트리밍 이벤트 수신
# PCM(24kHz, 16bit mono) -> WAV 저장

## 실습 문제

### 문제 1 — 나만의 오디오북 만들기
좋아하는 문장이나 짧은 글을 골라 `audio.speech`(`gpt-4o-mini-tts`)로 음성을 생성하세요.
- `voice` 를 바꿔보고, `instructions` 로 어조(예: "차분한 다큐멘터리 내레이션")를 지정해 보세요.
- 생성한 mp3를 `Audio` 로 재생해 확인하세요.

### 문제 2 — 받아쓰기 검증
문제 1에서 만든 음성을 `audio.transcriptions`(`gpt-4o-transcribe`)로 다시 텍스트로 변환하고,
원본 텍스트와 얼마나 일치하는지 비교해 보세요.

### 테스트 입력 예시
- 텍스트: `"인공지능은 인간의 가능성을 확장하는 도구입니다."`
- instructions: `"느리고 또박또박, 강연자처럼 말하세요."`